# BTC Cycle Investment Model

Notebook limpio para visualizar Bitcoin como un oscilador de ciclos.

**Fase actual:** carga robusta del dataset + oscilador histórico modularizado.  
**No incluye todavía:** ensemble de modelos futuros, Monte Carlo ni probabilidades de entrada.


In [ ]:
# ============================================================
# 0. Imports y configuración
# ============================================================

from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# -----------------------------
# Dataset
# -----------------------------

DATA_CANDIDATE_PATHS = [
    Path("coin-metrics-new-chart.tsv.tsv"),
    Path("../coin-metrics-new-chart.tsv.tsv"),
    Path("/content/coin-metrics-new-chart.tsv.tsv"),
    Path("/content/BTC.tsv.tsv"),
    Path("/content/btc-cycle-model/coin-metrics-new-chart.tsv.tsv"),
    Path("/content/btc-cycle-model-main/coin-metrics-new-chart.tsv.tsv"),
]

# Fallback útil cuando abres el notebook directamente desde GitHub en Colab.
# Colab no descarga automáticamente el dataset del repo, así que esta URL lo resuelve.
DATA_CANDIDATE_URLS = [
    "https://raw.githubusercontent.com/AdriaLazaro/btc-cycle-model/main/coin-metrics-new-chart.tsv.tsv"
]

POSSIBLE_ENCODINGS = ["utf-16", "utf-8", "latin1"]

# -----------------------------
# Parámetros del oscilador
# -----------------------------

START_DATE = "2013-01-01"
RESAMPLE_RULE = "3D"       # "D" = diario, "3D" = más pulso, "W" = más suave
ROLLING_SMOOTH = 1         # 1 = sin suavizado adicional

DRAWDOWN_SUAVE = 0.66
DRAWDOWN_DURO = 0.76
MULT_MIN = 3.0
MULT_MAX = 6.0
FUTURE_END = "2031-03-01"

FIGSIZE = (18, 8)

COLORS = {
    "cycle": "#3f73b5",
    "raw": "#f2a65a",
    "trend": "#3f73b5",
    "upper": "#ff7b72",
    "lower": "#2fbf71",
    "min_window": "#f6bd60",
    "max_window": "#c77dff",
    "today": "#f28e2b",
}

CYCLE_SPANS = [
    (pd.Timestamp("2013-11-01"), pd.Timestamp("2017-12-31"), "#eaf4ff", "Ciclo 2013-2017"),
    (pd.Timestamp("2017-12-31"), pd.Timestamp("2021-11-30"), "#fff3e6", "Ciclo 2017-2021"),
    (pd.Timestamp("2021-11-30"), pd.Timestamp("2025-12-31"), "#edf8ee", "Ciclo 2021-2025"),
    (pd.Timestamp("2025-12-31"), pd.Timestamp(FUTURE_END), "#f5edff", "Ciclo proyectado"),
]

In [ ]:
# ============================================================
# 1. Funciones de carga y preparación
# ============================================================

def load_btc_data(candidate_paths=DATA_CANDIDATE_PATHS,
                  candidate_urls=DATA_CANDIDATE_URLS,
                  encodings=POSSIBLE_ENCODINGS):
    """
    Carga el dataset BTC/USD desde rutas locales habituales o desde GitHub raw.
    Prueba varias codificaciones porque el TSV de Coin Metrics suele venir en UTF-16.
    """
    errors = []

    # 1) Rutas locales
    for path in candidate_paths:
        if not path.exists():
            errors.append(f"Missing local path: {path}")
            continue

        for encoding in encodings:
            try:
                df = pd.read_csv(path, sep="\t", encoding=encoding)
                return df, str(path), encoding
            except Exception as exc:
                errors.append(f"{path} with {encoding}: {type(exc).__name__}: {exc}")

    # 2) URLs remotas
    for url in candidate_urls:
        for encoding in encodings:
            try:
                df = pd.read_csv(url, sep="\t", encoding=encoding)
                return df, url, encoding
            except Exception as exc:
                errors.append(f"{url} with {encoding}: {type(exc).__name__}: {exc}")

    raise FileNotFoundError(
        "No se pudo cargar el dataset BTC.\n"
        "Coloca `coin-metrics-new-chart.tsv.tsv` en la raíz del repo, "
        "o súbelo a `/content/` en Colab.\n\n"
        "Intentos realizados:\n- " + "\n- ".join(errors[-12:])
    )


def prepare_btc_data(df, start_date=START_DATE, resample_rule=RESAMPLE_RULE):
    """
    Limpia columnas, convierte fechas/precios y crea una serie regular.
    Usamos log-price porque BTC crece de forma multiplicativa, no lineal.
    """
    df = df.copy()

    df = df.rename(columns={
        "Time": "date",
        "BTC / USD Denominated Closing Price": "price"
    })

    required = {"date", "price"}
    missing = required - set(df.columns)
    if missing:
        raise ValueError(f"Faltan columnas necesarias: {missing}. Columnas detectadas: {list(df.columns)}")

    df["date"] = pd.to_datetime(df["date"])
    df["price"] = pd.to_numeric(df["price"], errors="coerce")

    df = (
        df.dropna(subset=["date", "price"])
          .sort_values("date")
          .drop_duplicates("date")
    )

    btc = df[df["date"] >= start_date].copy()

    if resample_rule is not None:
        btc = (
            btc.set_index("date")
               .resample(resample_rule)
               .last()
               .ffill()
               .dropna()
               .reset_index()
        )

    btc["log_price"] = np.log(btc["price"])
    return btc


raw_df, DATA_PATH_USED, DATA_ENCODING_USED = load_btc_data()
btc = prepare_btc_data(raw_df)

print(f"Dataset path: {DATA_PATH_USED}")
print(f"Encoding: {DATA_ENCODING_USED}")
print(f"Rows: {len(raw_df):,}")
print(f"Dates: {raw_df.iloc[:, 0].min()} -> {raw_df.iloc[:, 0].max()}")
print(f"Columns: {list(raw_df.columns)}")
print(f"Prepared rows from {START_DATE}: {len(btc):,}")

btc.head()

In [ ]:
# ============================================================
# 2. Funciones del oscilador
# ============================================================

def year_diff(d2, d1):
    """Diferencia temporal en años."""
    return (pd.Timestamp(d2) - pd.Timestamp(d1)).days / 365.25


def get_turning_point(data, start, end, kind, label, phase):
    """
    Encuentra un máximo o mínimo real dentro de una ventana histórica.
    La fase se asigna manualmente para representar la intensidad del ciclo.
    """
    subset = data[(data["date"] >= start) & (data["date"] <= end)].copy()

    if len(subset) == 0:
        raise ValueError(f"No hay datos entre {start} y {end} para {label}")

    if kind == "max":
        row = subset.loc[subset["price"].idxmax()]
    elif kind == "min":
        row = subset.loc[subset["price"].idxmin()]
    else:
        raise ValueError("kind debe ser 'max' o 'min'")

    return {
        "date": row["date"],
        "price": row["price"],
        "log_price": np.log(row["price"]),
        "label": label,
        "phase": phase,
        "kind": kind
    }


def build_turning_points(data):
    """
    Turning points principales desde 2013.
    Estos puntos definen la normalización del oscilador por tramos.
    """
    turns = [
        get_turning_point(data, "2013-01-01", "2013-04-30", "min", "Inicio 2013", -0.35),

        get_turning_point(data, "2013-10-01", "2014-01-31", "max", "Max 2013",  1.00),
        get_turning_point(data, "2014-12-01", "2015-03-31", "min", "Min 2015", -0.75),

        get_turning_point(data, "2017-11-01", "2018-01-31", "max", "Max 2017",  0.75),
        get_turning_point(data, "2018-11-01", "2019-02-28", "min", "Min 2018", -0.50),

        get_turning_point(data, "2021-10-01", "2021-12-31", "max", "Max 2021",  0.50),
        get_turning_point(data, "2022-10-01", "2023-01-31", "min", "Min 2022", -0.35),

        get_turning_point(data, "2025-01-01", "2026-03-01", "max", "Max 2025",  0.35),
    ]

    return pd.DataFrame(turns).sort_values("date").reset_index(drop=True)


def compute_damping_parameters(turns):
    """
    Ajusta amortiguamiento exponencial a picos y valles.
    Menos amortiguamiento = amplitud futura más alta.
    Más amortiguamiento = amplitud futura más baja.
    """
    peaks = turns[turns["kind"] == "max"].copy()
    troughs = turns[(turns["kind"] == "min") & (turns["label"] != "Inicio 2013")].copy()

    peak_dates = peaks["date"].to_list()
    peak_amps = peaks["phase"].abs().to_numpy()

    trough_dates = troughs["date"].to_list()
    trough_amps = troughs["phase"].abs().to_numpy()

    peak_betas = []
    for i in range(len(peak_amps) - 1):
        dt = year_diff(peak_dates[i + 1], peak_dates[i])
        peak_betas.append(-np.log(peak_amps[i + 1] / peak_amps[i]) / dt)

    trough_betas = []
    for i in range(len(trough_amps) - 1):
        dt = year_diff(trough_dates[i + 1], trough_dates[i])
        trough_betas.append(-np.log(trough_amps[i + 1] / trough_amps[i]) / dt)

    peak_betas = np.array(peak_betas)
    trough_betas = np.array(trough_betas)

    params = {
        "peaks": peaks,
        "troughs": troughs,
        "beta_peak_mean": peak_betas.mean(),
        "beta_peak_std": peak_betas.std(ddof=1),
        "beta_trough_mean": trough_betas.mean(),
        "beta_trough_std": trough_betas.std(ddof=1),
    }

    params["beta_peak_low"] = max(0.001, params["beta_peak_mean"] - params["beta_peak_std"])
    params["beta_peak_high"] = params["beta_peak_mean"] + params["beta_peak_std"]
    params["beta_trough_low"] = max(0.001, params["beta_trough_mean"] - params["beta_trough_std"])
    params["beta_trough_high"] = params["beta_trough_mean"] + params["beta_trough_std"]

    return params


def estimate_next_cycle_windows(turns, damping_params,
                                drawdown_suave=DRAWDOWN_SUAVE,
                                drawdown_duro=DRAWDOWN_DURO,
                                mult_min=MULT_MIN,
                                mult_max=MULT_MAX):
    """Estima ventanas temporales y rangos de precio del próximo mínimo/máximo."""
    peaks = damping_params["peaks"]
    troughs = damping_params["troughs"]

    peak_dates = peaks["date"].to_list()

    peak_to_peak = np.array([
        year_diff(peak_dates[i + 1], peak_dates[i])
        for i in range(len(peak_dates) - 1)
    ])

    historical_peak_to_trough = np.array([
        year_diff(troughs.iloc[0]["date"], peaks.iloc[0]["date"]),
        year_diff(troughs.iloc[1]["date"], peaks.iloc[1]["date"]),
        year_diff(troughs.iloc[2]["date"], peaks.iloc[2]["date"]),
    ])

    pp_mean = peak_to_peak.mean()
    pp_std = peak_to_peak.std(ddof=1)

    pt_mean = historical_peak_to_trough.mean()
    pt_std = historical_peak_to_trough.std(ddof=1)

    last_peak = peaks.iloc[-1]
    last_peak_date = last_peak["date"]
    last_peak_price = last_peak["price"]

    next_min_early = last_peak_date + pd.to_timedelta((pt_mean - pt_std) * 365.25, unit="D")
    next_min_late = last_peak_date + pd.to_timedelta((pt_mean + pt_std) * 365.25, unit="D")
    next_min_center = last_peak_date + pd.to_timedelta(pt_mean * 365.25, unit="D")

    next_max_early = last_peak_date + pd.to_timedelta((pp_mean - pp_std) * 365.25, unit="D")
    next_max_late = last_peak_date + pd.to_timedelta((pp_mean + pp_std) * 365.25, unit="D")
    next_max_center = last_peak_date + pd.to_timedelta(pp_mean * 365.25, unit="D")

    min_price_low = last_peak_price * (1 - drawdown_duro)
    min_price_high = last_peak_price * (1 - drawdown_suave)

    max_price_low = min_price_low * mult_min
    max_price_high = min_price_high * mult_max

    return {
        "pp_mean": pp_mean,
        "pp_std": pp_std,
        "pt_mean": pt_mean,
        "pt_std": pt_std,
        "last_peak_date": last_peak_date,
        "last_peak_price": last_peak_price,
        "next_min_early": next_min_early,
        "next_min_late": next_min_late,
        "next_min_center": next_min_center,
        "next_max_early": next_max_early,
        "next_max_late": next_max_late,
        "next_max_center": next_max_center,
        "min_price_low": min_price_low,
        "min_price_high": min_price_high,
        "max_price_low": max_price_low,
        "max_price_high": max_price_high,
    }


def compute_envelopes(damping_params, future_end=FUTURE_END, n_points=700):
    """Calcula bandas exponenciales superior e inferior."""
    peaks = damping_params["peaks"]
    troughs = damping_params["troughs"]

    env_dates = pd.date_range(peaks.iloc[0]["date"], pd.Timestamp(future_end), periods=n_points)

    upper_anchor_date = peaks.iloc[0]["date"]
    upper_anchor_amp = abs(peaks.iloc[0]["phase"])

    lower_anchor_date = troughs.iloc[0]["date"]
    lower_anchor_amp = abs(troughs.iloc[0]["phase"])

    upper_t = np.array([max(0, year_diff(d, upper_anchor_date)) for d in env_dates])
    lower_t = np.array([max(0, year_diff(d, lower_anchor_date)) for d in env_dates])

    upper_less_damp = upper_anchor_amp * np.exp(-damping_params["beta_peak_low"] * upper_t)
    upper_more_damp = upper_anchor_amp * np.exp(-damping_params["beta_peak_high"] * upper_t)

    lower_less_damp = -lower_anchor_amp * np.exp(-damping_params["beta_trough_low"] * lower_t)
    lower_more_damp = -lower_anchor_amp * np.exp(-damping_params["beta_trough_high"] * lower_t)

    return {
        "env_dates": env_dates,
        "upper_less_damp": upper_less_damp,
        "upper_more_damp": upper_more_damp,
        "lower_less_damp": lower_less_damp,
        "lower_more_damp": lower_more_damp,
    }


def build_cycle_oscillator(data, turns, windows, damping_params, rolling_smooth=ROLLING_SMOOTH):
    """
    Construye el oscilador detallado.
    Normalizamos cada tramo en log-price entre turning points para conservar los pulsos reales.
    """
    troughs = damping_params["troughs"]
    last_trough = troughs.iloc[-1]

    dt_next_trough = year_diff(windows["next_min_center"], last_trough["date"])

    next_trough_amp_less = abs(last_trough["phase"]) * np.exp(-damping_params["beta_trough_low"] * dt_next_trough)
    next_trough_amp_more = abs(last_trough["phase"]) * np.exp(-damping_params["beta_trough_high"] * dt_next_trough)
    next_trough_phase = -np.mean([next_trough_amp_less, next_trough_amp_more])

    projected_min = {
        "date": windows["next_min_center"],
        "price": (windows["min_price_low"] + windows["min_price_high"]) / 2,
        "log_price": np.log((windows["min_price_low"] + windows["min_price_high"]) / 2),
        "label": "Min proyectado",
        "phase": next_trough_phase,
        "kind": "min"
    }

    turns_for_mapping = pd.concat([
        turns,
        pd.DataFrame([projected_min])
    ]).sort_values("date").reset_index(drop=True)

    segments = []

    for i in range(len(turns_for_mapping) - 1):
        start = turns_for_mapping.iloc[i]
        end = turns_for_mapping.iloc[i + 1]

        seg_end_date = min(end["date"], data["date"].max())
        seg = data[(data["date"] >= start["date"]) & (data["date"] <= seg_end_date)].copy()

        if len(seg) == 0:
            continue

        log_start = start["log_price"]
        log_end = end["log_price"]
        phase_start = start["phase"]
        phase_end = end["phase"]

        if abs(log_end - log_start) < 1e-9:
            continue

        seg["ratio"] = (seg["log_price"] - log_start) / (log_end - log_start)
        seg["cycle_raw"] = phase_start + seg["ratio"] * (phase_end - phase_start)

        # Clipping suave: deja rebotes reales, pero evita que rompan la fase del tramo.
        low = min(phase_start, phase_end) - 0.12
        high = max(phase_start, phase_end) + 0.12
        seg["cycle_raw"] = seg["cycle_raw"].clip(low, high)

        segments.append(seg)

    cycle = pd.concat(segments).sort_values("date").drop_duplicates("date")
    cycle["cycle_smooth"] = (
        cycle["cycle_raw"]
        .rolling(rolling_smooth, center=True, min_periods=1)
        .mean()
    )

    return cycle, turns_for_mapping

In [ ]:
# ============================================================
# 3. Gráfico principal
# ============================================================

def plot_cycle_oscillator(data, turns, cycle, envelopes, windows, figsize=FIGSIZE):
    """Dibuja el oscilador histórico con envolventes y ventanas futuras."""
    fig, ax = plt.subplots(figsize=figsize)

    # Fondos pastel por ciclo
    for start, end, color, label in CYCLE_SPANS:
        ax.axvspan(start, end, color=color, alpha=0.45, zorder=0)

    # Oscilador con detalle real
    ax.plot(
        cycle["date"],
        cycle["cycle_smooth"],
        linewidth=2.8,
        color=COLORS["cycle"],
        label="Ciclo BTC normalizado con datos reales",
        zorder=4
    )

    ax.plot(
        cycle["date"],
        cycle["cycle_raw"],
        linewidth=0.7,
        alpha=0.25,
        color=COLORS["raw"],
        label="Detalle real",
        zorder=3
    )

    ax.axhline(
        0,
        linewidth=2.8,
        color=COLORS["trend"],
        alpha=0.9,
        label="Tendencia normalizada",
        zorder=2
    )

    env_dates = envelopes["env_dates"]

    # Envolvente superior
    ax.fill_between(
        env_dates,
        envelopes["upper_more_damp"],
        envelopes["upper_less_damp"],
        color=COLORS["upper"],
        alpha=0.16,
        label="Envolvente superior ± amortiguamiento",
        zorder=1
    )
    ax.plot(env_dates, envelopes["upper_less_damp"], "--", color=COLORS["upper"], linewidth=2, zorder=5)
    ax.plot(env_dates, envelopes["upper_more_damp"], "--", color=COLORS["upper"], linewidth=2, zorder=5)

    # Envolvente inferior
    ax.fill_between(
        env_dates,
        envelopes["lower_less_damp"],
        envelopes["lower_more_damp"],
        color=COLORS["lower"],
        alpha=0.16,
        label="Envolvente inferior ± amortiguamiento",
        zorder=1
    )
    ax.plot(env_dates, envelopes["lower_less_damp"], "--", color=COLORS["lower"], linewidth=2, zorder=5)
    ax.plot(env_dates, envelopes["lower_more_damp"], "--", color=COLORS["lower"], linewidth=2, zorder=5)

    # Ventanas futuro
    ax.axvspan(windows["next_min_early"], windows["next_min_late"],
               color=COLORS["min_window"], alpha=0.22, label="Ventana próximo mínimo", zorder=1)
    ax.axvline(windows["next_min_center"], color=COLORS["min_window"], linestyle=":", linewidth=3, zorder=6)

    ax.axvspan(windows["next_max_early"], windows["next_max_late"],
               color=COLORS["max_window"], alpha=0.18, label="Ventana próximo máximo", zorder=1)
    ax.axvline(windows["next_max_center"], color=COLORS["max_window"], linestyle=":", linewidth=3, zorder=6)

    # Puntos y etiquetas clave
    ax.scatter(turns["date"], turns["phase"], s=70, color=COLORS["cycle"], zorder=7)

    for _, row in turns.iterrows():
        text = f"{row['label']}\n${row['price']:,.0f}"
        offset = 12 if row["phase"] >= 0 else -28
        va = "bottom" if row["phase"] >= 0 else "top"

        ax.annotate(
            text,
            (row["date"], row["phase"]),
            textcoords="offset points",
            xytext=(0, offset),
            ha="center",
            va=va,
            fontsize=8.5,
            zorder=8
        )

    # Punto actual
    last = data.iloc[-1]
    last_cycle = cycle.iloc[-1]

    ax.scatter(last_cycle["date"], last_cycle["cycle_smooth"],
               s=90, color=COLORS["today"], zorder=8)

    ax.annotate(
        f"Hoy\n${last['price']:,.0f}",
        (last_cycle["date"], last_cycle["cycle_smooth"]),
        textcoords="offset points",
        xytext=(12, -8),
        ha="left",
        va="center",
        fontsize=9,
        zorder=9
    )

    # Cajas de texto futuro
    min_text = (
        "Próximo mínimo\n"
        f"{windows['next_min_early'].strftime('%b %Y')} – {windows['next_min_late'].strftime('%b %Y')}\n"
        f"${windows['min_price_low']:,.0f} – ${windows['min_price_high']:,.0f}"
    )

    max_text = (
        "Próximo máximo\n"
        f"{windows['next_max_early'].strftime('%b %Y')} – {windows['next_max_late'].strftime('%b %Y')}\n"
        f"${windows['max_price_low']:,.0f} – ${windows['max_price_high']:,.0f}"
    )

    ax.text(
        windows["next_min_center"],
        -0.92,
        min_text,
        ha="center",
        va="top",
        fontsize=10,
        bbox=dict(boxstyle="round,pad=0.35", facecolor="white", edgecolor=COLORS["min_window"], alpha=0.96),
        zorder=10
    )

    ax.text(
        windows["next_max_center"],
        0.82,
        max_text,
        ha="center",
        va="bottom",
        fontsize=10,
        bbox=dict(boxstyle="round,pad=0.35", facecolor="white", edgecolor=COLORS["max_window"], alpha=0.96),
        zorder=10
    )

    # Etiquetas visuales de ciclo
    cycle_labels = [
        (pd.Timestamp("2015-12-01"), "Ciclo 2013-2017"),
        (pd.Timestamp("2019-11-01"), "Ciclo 2017-2021"),
        (pd.Timestamp("2023-11-01"), "Ciclo 2021-2025"),
        (pd.Timestamp("2028-02-01"), "Ciclo proyectado"),
    ]

    for x_pos, label in cycle_labels:
        ax.text(x_pos, -1.08, label, ha="center", va="bottom", fontsize=9, alpha=0.65)

    ax.set_yticks([-1.0, -0.5, 0.0, 0.5, 1.0])
    ax.set_yticklabels(["Capitulación", "Infravalorado", "Tendencia", "Sobrecalentado", "Euforia"])

    ax.set_ylim(-1.15, 1.15)
    ax.set_xlim(pd.Timestamp(START_DATE), pd.Timestamp(FUTURE_END))

    ax.set_title(
        "Bitcoin: oscilador de ciclos desde 2013 con detalle real\n"
        "y envolventes exponenciales de amortiguamiento",
        fontsize=15
    )

    ax.set_xlabel("Fecha")
    ax.set_ylabel("Fase del ciclo")
    ax.grid(True, axis="y", alpha=0.3)

    ax.legend(loc="upper left", bbox_to_anchor=(1.01, 1.0), fontsize=9, frameon=True)
    plt.tight_layout(rect=[0, 0, 0.82, 1])
    plt.show()


turns = build_turning_points(btc)
damping_params = compute_damping_parameters(turns)
windows = estimate_next_cycle_windows(turns, damping_params)
envelopes = compute_envelopes(damping_params)
cycle, turns_for_mapping = build_cycle_oscillator(btc, turns, windows, damping_params)

plot_cycle_oscillator(btc, turns, cycle, envelopes, windows)

In [ ]:
# ============================================================
# 4. Resumen numérico
# ============================================================

current_date = btc["date"].iloc[-1]
current_price = btc["price"].iloc[-1]

print("Resumen del BTC Cycle Model")
print("-" * 40)
print(f"Fecha actual del dataset: {current_date.date()}")
print(f"Precio actual BTC: ${current_price:,.0f}")
print()
print("Amortiguamiento superior:")
print(f"  Menos amortiguamiento β: {damping_params['beta_peak_low']:.4f} / año")
print(f"  Más amortiguamiento β:   {damping_params['beta_peak_high']:.4f} / año")
print()
print("Amortiguamiento inferior:")
print(f"  Menos amortiguamiento β: {damping_params['beta_trough_low']:.4f} / año")
print(f"  Más amortiguamiento β:   {damping_params['beta_trough_high']:.4f} / año")
print()
print(f"Periodo pico→pico medio: {windows['pp_mean']:.2f} ± {windows['pp_std']:.2f} años")
print(f"Tiempo pico→mínimo medio: {windows['pt_mean']:.2f} ± {windows['pt_std']:.2f} años")
print()
print("Proyección:")
print(
    f"Próximo mínimo: {windows['next_min_early'].strftime('%Y-%m-%d')} "
    f"a {windows['next_min_late'].strftime('%Y-%m-%d')}"
)
print(f"Precio mínimo estimado: ${windows['min_price_low']:,.0f} – ${windows['min_price_high']:,.0f}")
print()
print(
    f"Próximo máximo: {windows['next_max_early'].strftime('%Y-%m-%d')} "
    f"a {windows['next_max_late'].strftime('%Y-%m-%d')}"
)
print(f"Precio máximo estimado: ${windows['max_price_low']:,.0f} – ${windows['max_price_high']:,.0f}")